# 07b. LLM 2차 시도 — 역할 축소 (D61, 2026-11-19)

- 1차(07): LLM이 112건 중 111건을 결함으로 판정해 결함을 가려내지 못했다.
- 2차: 범위 안/밖·사유 코드·원천 등급 비교는 규칙(03) 결과를 그대로 쓴다. LLM은 (a) 고객 등급 언급 문장 분류, (b) answer의 이관 표현 여부만 판단한다.
- 결함 = 규칙의 범위 판단 + LLM 분류를 코드로 결합 (`src/rag/classify.combine`). 프롬프트에 규칙 판정을 넣지 않는다.
- 검색: 사례마다 자기 값(성향·언급 등급)으로 질의. 색인에 한국투자증권 준칙(항 단위)·금소법·표준투자권유준칙 포함. 회사 필터 버그 수정.
- 예시(few-shot) 5개는 검증 표본 밖 CoT에서 골랐다 (`src/rag/fewshot.csv`). 일반 표현 예시는 적당한 것이 없어 지시문 설명만 둔다.
- 기존 희망 40·이관 20은 분류 정확도만 보고하며, 이 결과를 보고 프롬프트를 고치지 않는다.

데이터: AI Hub(한국지능정보사회진흥원) 사업 결과

In [ ]:
import sys, json, time, re
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
from src.rag import assert_local
from src.rag import regulations, index as rindex
from src.rag.classify import classify, combine, fewshot_messages, mention_sentences, query_for, _llm
from src.load_cot import load_cot, load_products
from src.judge import judge
from src.transfer import load_terms, question_mentions
assert_local()
AS_OF, COMPANY = "2026-11-19", "한국투자증권"
RERUN = False
CACHE_DIR = ROOT / "data" / "llm_cache"; OUT_DIR = ROOT / "outputs" / "llm"; OUT_DIR.mkdir(parents=True, exist_ok=True)
cot = load_cot(); prod = load_products()
j, ctx = judge(cot, prod, as_of="2026-11-18")
terms = load_terms()
rd = lambda p: pd.read_csv(ROOT / "data" / "review" / p, encoding="utf-8-sig", dtype=str, keep_default_na=False)

## 1. 색인 (한국투자증권 준칙 항 단위 + 금소법 + 표준투자권유준칙 + 상품 문서)

In [ ]:
chunks = regulations.load_chunks()
print(chunks.groupby(["층위", "doc_id"]).size().to_string())
t0 = time.time(); store = rindex.build(chunks); print(f"색인 {len(chunks)} 조각, {time.time() - t0:.0f}초")

## 2. 새 검증 표본 → `data/review/LLM검수표.csv`

- 결함 15: 규칙 이관 판정이 '결함'인 CoT에서 무작위. 결함 아님 15: 성향 밖 회피 5 + 성향 안 언급 10 (규칙 이관 판정 '해당 없음').
- 뺀 CoT: 범위밖 32, 희망 40, 이관 20 (기존 112건 사례의 CoT 전부 포함), few-shot 예시 5.
- 열: cot_id, 분할, 성향, 허용 등급, 언급 등급, 언급 문장, 추천 등급, answer, 내 분류, 내 이관 표현, 내 판정 (판정 칸은 비워 둔다). 규칙·LLM 결과는 넣지 않는다.
- 이미 있으면 덮어쓰지 않는다. 원문이 있어 git 제외 폴더에 둔다.

In [ ]:
used = set()
for f in ["범위밖_분류표.csv", "희망검수표.csv", "희망검수표2.csv", "이관검수표.csv"]:
    used |= set(rd(f)["cot_id"].astype(int))
used |= set(pd.read_csv(ROOT / "src" / "rag" / "fewshot.csv")["cot_id"])
pool = j[(j["이관 분류"] != "언급 없음") & ~j["cot_id"].isin(used)]
p_def = pool[pool["이관 판정"] == "결함"]
p_avoid = pool[(pool["이관 분류"] == "회피") & (pool["성향 밖 언급"] != "")]
p_in = pool[(pool["성향 밖 언급"] == "") & (pool["이관 판정"] == "해당 없음")]
print("모집단 — 결함:", len(p_def), "/ 성향 밖 회피:", len(p_avoid), "/ 성향 안:", len(p_in))
new = pd.concat([p_def.sample(15, random_state=0), p_avoid.sample(5, random_state=0), p_in.sample(10, random_state=0)])
c, rec = ctx["cots"], ctx["recs"]
rec_g = rec.groupby("cot_id")["json_grade"].agg(lambda g: "|".join(str(int(x)) if pd.notna(x) else "?" for x in g))
rows = []
for cid in new["cot_id"]:
    x = c.loc[cid]
    qm = question_mentions(x["question"], terms)
    rows.append({"cot_id": cid, "분할": x["split"], "성향": x["성향_D21"],
                 "허용 등급": "-".join(map(str, sorted(ctx["range"].get(x["성향_D21"], [])))),
                 "언급 등급": ",".join(map(str, sorted(qm["언급 등급"]))), "언급 문장": " / ".join(mention_sentences(x["question"])),
                 "추천 등급": rec_g.get(cid, ""), "answer": x["answer"], "내 분류": "", "내 이관 표현": "", "내 판정": ""})
sheet = pd.DataFrame(rows).sort_values("cot_id")
p8 = ROOT / "data" / "review" / "LLM검수표.csv"
if p8.exists():
    print("이미 있어 덮어쓰지 않음:", p8)
else:
    sheet.to_csv(p8, index=False, encoding="utf-8-sig"); print("저장:", p8, len(sheet))
print(sheet[["cot_id", "분할", "성향", "언급 등급", "추천 등급"]].to_string(index=False))

## 3. 사례 구성과 LLM 분류 (새 30 + 기존 희망 40 + 이관 20)

In [ ]:
h1, h2, t7, s8 = rd("희망검수표.csv"), rd("희망검수표2.csv"), rd("이관검수표.csv"), rd("LLM검수표.csv")
targets = ([("희망1", int(x)) for x in h1["cot_id"]] + [("희망2", int(x)) for x in h2["cot_id"]] +
           [("이관20", int(x)) for x in t7["cot_id"]] + [("LLM30", int(x)) for x in s8["cot_id"]])

def make_case(cid):
    x = c.loc[cid]; rng = ctx["range"].get(x["성향_D21"], set())
    qm = question_mentions(x["question"], terms)
    outside = set(qm.loc[~qm["해석 등급"].isin(rng), "해석 등급"]) if rng else set()
    literal = set(qm.loc[(qm["분류"] == "R4 비교 표현") & ~qm["언급 등급"].isin(rng), "언급 등급"]) if rng else set()
    grades = sorted(outside or set(qm["언급 등급"]))
    return {"cot_id": cid, "성향": x["성향_D21"], "허용 등급": "-".join(map(str, sorted(rng))) or "알 수 없음",
            "문장": mention_sentences(x["question"]), "answer": str(x["answer"]),
            "질의": query_for(x["성향_D21"], grades), "_outside": outside, "_literal": literal,
            "_rec": set(rec.loc[rec["cot_id"] == cid, "json_grade"].dropna().astype(int))}
cases = {cid: make_case(cid) for _, cid in targets}
print("사례:", len(cases), "/ 표본별:", pd.Series([t for t, _ in targets]).value_counts().to_dict())

In [ ]:
cache = CACHE_DIR / "classify_results.csv"
if cache.exists() and not RERUN:
    res = pd.read_csv(cache, encoding="utf-8-sig", keep_default_na=False); print("캐시 사용:", len(res))
else:
    # GPU 8GB: 검색(bge-m3)을 먼저 모두 끝내고 bge-m3를 내린 뒤 LLM을 올린다 (1차 실행에서 두 모델이 겹쳐 CUDA 오류로 멈춤)
    evs = {cid: rindex.search(store, cases[cid]["질의"], AS_OF, COMPANY, k=5) for cid in cases}
    import httpx
    from src.rag import OLLAMA_URL
    httpx.post(f"{OLLAMA_URL}/api/embed", json={"model": "bge-m3", "input": "", "keep_alive": 0}, timeout=60)
    print("검색 완료, bge-m3 내림:", httpx.get(f"{OLLAMA_URL}/api/ps").json().get("models", []) and
          [m["name"] for m in httpx.get(f"{OLLAMA_URL}/api/ps").json()["models"]])
    llm = _llm(); fs = fewshot_messages(cot); rows = []; t0 = time.time()
    for i, cid in enumerate(cases):
        ev = evs[cid]
        rows.append(classify(cases[cid], ev, fs, llm))
        if (i + 1) % 30 == 0: print(i + 1, "/", len(cases), f"{time.time() - t0:.0f}초")
    res = pd.DataFrame(rows); res.to_csv(cache, index=False, encoding="utf-8-sig")
res["LLM 이관 표현"] = res["LLM 이관 표현"].astype(str).str.lower().eq("true")
res["결합 판정"] = [combine(r["LLM 분류"] if r["상태"] == "정상" else None, r["LLM 이관 표현"],
                       cases[r["cot_id"]]["_outside"], cases[r["cot_id"]]["_literal"], cases[r["cot_id"]]["_rec"])
                for r in res.to_dict("records")]
print("상태:", res["상태"].value_counts().to_dict(), "/ 재시도 있음:", (res["재시도"] > 0).sum(),
      f"/ 평균 {res['소요(초)'].mean():.1f}초")
print("LLM 분류:", res["LLM 분류"].value_counts().to_dict(), "/ 이관 표현 true:", res["LLM 이관 표현"].sum())
print("결합 판정:", res["결합 판정"].value_counts().to_dict())

## 4. 기존 희망 40 + 이관 20: LLM 분류 정확도와 결함 일치 (프롬프트 수정에 쓰지 않음)

- 사람 라벨 → LLM 분류: 희망(명확)→희망, 고려 중, 경험, 비교 표현, 회피, 오탐(일반 표현)→일반 표현.
- 사람 결함: 희망검수표2 20건 결함, 희망검수표1 20건 결함 아님, 이관검수표 '내 판정'.
- 규칙(03)의 이관 판정과 나란히 둔다.

In [ ]:
MAP = {"희망(명확)": "희망", "고려 중": "고려 중", "경험": "경험", "비교 표현": "비교 표현", "회피": "회피", "오탐(일반 표현)": "일반 표현"}
hum = pd.concat([
    pd.DataFrame({"표본": "희망1", "cot_id": h1["cot_id"].astype(int), "사람 분류": h1["내 판정"].map(MAP), "사람 결함": False}),
    pd.DataFrame({"표본": "희망2", "cot_id": h2["cot_id"].astype(int), "사람 분류": h2["내 판정"].map(MAP), "사람 결함": True}),
    pd.DataFrame({"표본": "이관20", "cot_id": t7["cot_id"].astype(int), "사람 분류": t7["내 분류"].map(MAP), "사람 결함": t7["내 판정"].eq("결함")}),
])
m = hum.merge(res, on="cot_id").merge(j[["cot_id", "이관 판정"]], on="cot_id")
m["분류 일치"] = m["사람 분류"] == m["LLM 분류"]
m["LLM 결함 일치"] = (m["결합 판정"] == "결함") == m["사람 결함"]
m["규칙 결함 일치"] = (m["이관 판정"] == "결함") == m["사람 결함"]
t = m.groupby("표본", sort=False).agg(크기=("cot_id", "size"), 분류_일치=("분류 일치", "sum"),
                                     LLM결합_결함_일치=("LLM 결함 일치", "sum"), 규칙_결함_일치=("규칙 결함 일치", "sum"))
t.loc["합계"] = t.sum()
print(t.to_string()); print()
print("[분류 혼동표] 행=사람, 열=LLM"); print(pd.crosstab(m["사람 분류"], m["LLM 분류"], margins=True)); print()
print("[결함] 행=사람, 열=LLM 결합"); print(pd.crosstab(m["사람 결함"], m["결합 판정"], margins=True)); print()
print("[확신도별 분류 일치]"); print(m.groupby("확신도")["분류 일치"].agg(["size", "sum"]).to_string())

## 5. 검색 적중 (이관 정답 조항이 상위 3개 안) · 인용 · 형식

In [ ]:
gold = pd.read_csv(ROOT / "src" / "rag" / "gold_articles.csv", encoding="utf-8", dtype=str, keep_default_na=False)
gi = gold[(gold["유형"] == "이관") & (gold["조항"] != "")]
gids = [f"{d}:{a}" for d, a in zip(gi["doc_id"], gi["조항"])]
res["정답 top3"] = res["검색 top3"].map(lambda s: any(t == g or t.startswith(g + "~") for t in str(s).split("|") for g in gids))
print("정답 조항:", gids)
print(f"이관 정답 조항이 top3 안에 있는 사례: {res['정답 top3'].sum()} / {len(res)}")
for g in gids:
    print(f"  {g}: {res['검색 top3'].str.contains(re.escape(g)).sum()}")
print("top3 조합 상위:"); print(res["검색 top3"].value_counts().head(5).to_string())
ok = res[res["상태"] == "정상"]
print(f"인용 조항이 검색 결과 안: {(~ok['인용 오류'].astype(str).str.lower().eq('true')).mean() * 100:.1f}% / 인용 없음(빈 목록): {(ok['인용 조항'] == '').sum()}")
print("JSON 첫 시도 실패:", (res["재시도"] > 0).sum(), "/ 재시도 후 실패:", (res["상태"] != "정상").sum(), "/", len(res))

## 6. 새 30건 비교 (사람 판정 후 실행)

- 판단 기준(미리 고정): **확정 대기** — 사용자 확정 후 이 칸과 D61에 적는다.

In [ ]:
s8 = rd("LLM검수표.csv")
if (s8["내 판정"] == "").all():
    print("아직 사람 판정 없음:", len(s8))
else:
    s8["cot_id"] = s8["cot_id"].astype(int)
    n = s8.merge(res, on="cot_id").merge(j[["cot_id", "이관 판정"]], on="cot_id")
    n["사람 결함"] = n["내 판정"] == "결함"
    print(pd.crosstab(n["사람 결함"], n["결합 판정"], margins=True))
    print(pd.crosstab(n["사람 결함"], n["이관 판정"], margins=True))
    na = n[~n["사람 결함"]]
    print(f"결함 아님 {len(na)}건 중 LLM 결합 '결함 아님': {(na['결합 판정'] == '결함 아님').sum()}")
    d = n[n["사람 결함"]]
    print(f"결함 {len(d)}건 중 LLM 결합 '결함': {(d['결합 판정'] == '결함').sum()}")
    if "내 분류" in n: print(pd.crosstab(n["내 분류"], n["LLM 분류"], margins=True))

In [ ]:
keep = ["cot_id", "상태", "재시도", "LLM 분류", "LLM 이관 표현", "확신도", "결합 판정", "인용 조항", "인용 오류", "검색 top3", "정답 top3", "소요(초)"]
res[keep].to_csv(OUT_DIR / "classify_results.csv", index=False, encoding="utf-8-sig")
t.to_csv(OUT_DIR / "classify_old60_summary.csv", encoding="utf-8-sig")
print("저장:", OUT_DIR / "classify_results.csv")

---
데이터: AI Hub(한국지능정보사회진흥원) 사업 결과